In [1]:
# ============================================================
# HealthBot - AI-Powered Patient Education Assistant
# Cell 1: Imports and Environment Setup
# ============================================================

import os
from typing import TypedDict, List, Dict, Any

from dotenv import load_dotenv
from langchain_community.tools.tavily_search import TavilySearchResults
from google import genai
from langgraph.graph import StateGraph, START, END

print("All required libraries imported successfully.")

C:\Users\91866\AppData\Local\Temp\ipykernel_38236\3830412342.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults


All required libraries imported successfully.


In [2]:
# ============================================================
# Cell 2: Load Configuration
# ============================================================

load_dotenv("config.env", override=True)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GEMINI_MODEL = os.getenv("GEMINI_MODEL")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

print("Configuration loaded.")
print("Gemini API key:", "FOUND" if GEMINI_API_KEY else "NOT FOUND")
print("Gemini model:", GEMINI_MODEL if GEMINI_MODEL else "NOT FOUND")
print("Tavily API key:", "FOUND" if TAVILY_API_KEY else "NOT FOUND")

Configuration loaded.
Gemini API key: FOUND
Gemini model: gemini-3.6-flash
Tavily API key: FOUND


In [3]:
# ============================================================
# Cell 3: Initialize API Clients
# ============================================================

gemini_client = genai.Client(api_key=GEMINI_API_KEY)

# Initialize the LangChain community tool for Tavily[cite: 2]
tavily_tool = TavilySearchResults(max_results=5)

print("Gemini client initialized successfully.")
print("Tavily tool initialized successfully.")

Gemini client initialized successfully.
Tavily tool initialized successfully.


C:\Users\91866\AppData\Local\Temp\ipykernel_38236\947597644.py:8: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_tool = TavilySearchResults(max_results=5)


In [4]:
# ============================================================
# Cell 4: Test Gemini API
# ============================================================

test_response = gemini_client.models.generate_content(
    model=GEMINI_MODEL,
    contents="Reply with exactly: GEMINI API WORKING"
)

print(test_response.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


GEMINI API WORKING


In [5]:
# ============================================================
# Cell 5: Test Tavily API
# ============================================================

test_search = tavily_tool.invoke({
    "query": "What are common symptoms of dehydration?"
})

print(test_search)

[{'title': 'Dehydration: MedlinePlus', 'url': 'https://medlineplus.gov/dehydration.html', 'content': "### What are the symptoms of dehydration?\n\nIn adults, the symptoms of dehydration include:\n\n Feeling very thirsty\n Dry mouth\n Urinating and sweating less than usual\n Dark-colored urine\n Dry skin\n Feeling tired\n Dizziness\n\nIn infants and young children, the symptoms of dehydration include:\n\n Dry mouth and tongue\n Crying without tears\n No wet diapers for 3 hours or more\n A high fever\n Being unusually sleepy or drowsy\n Irritability\n Eyes that look sunken\n\nDehydration can be mild, or it can be severe enough to be life-threatening. Get medical help right away if the symptoms also include:\n\n Confusion\n Fainting\n Lack of urination\n Rapid heartbeat\n Rapid breathing\n Shock\n\n### How is dehydration diagnosed?\n\nTo find out if you dehydration, your health care provider will: [...] You can become dehydrated because of:\n\n Diarrhea\n Vomiting\n Sweating too much\n Ur

In [6]:
# ============================================================
# Cell 6: Define HealthBot State
# ============================================================

class HealthBotState(TypedDict):
    user_query: str
    context: str
    search_results: List[Dict[str, Any]]
    response: str

print("HealthBot state defined successfully.")

HealthBot state defined successfully.


In [7]:
# ============================================================
# Cell 7: Tavily Search Node
# ============================================================

def search_medical_information(state: HealthBotState) -> HealthBotState:
    query = state["user_query"]

    # Invoke the LangChain Tavily tool[cite: 2]
    search_response = tavily_tool.invoke({
        "query": f"{query} symptoms causes treatment patient information"
    })

    # The LangChain tool directly returns a list of result dictionaries
    state["search_results"] = search_response

    return state


print("Tavily search node defined successfully.")

Tavily search node defined successfully.


In [8]:
# ============================================================
# Cell 8: Generate Patient-Friendly Health Summary
# ============================================================

def generate_response(state: HealthBotState) -> HealthBotState:

    # Get Tavily search results from the state
    search_results = state["search_results"]

    # Build context from Tavily search results
    context = "\n\n".join(
        [
            f"Source: {result.get('title', 'Unknown')}\n"
            f"URL: {result.get('url', 'Unknown')}\n"
            f"Content: {result.get('content', '')}"
            for result in search_results
        ]
    )

    # Save the Tavily context in state
    # This will be used later by the quiz and grading nodes.
    state["context"] = context

    # Prompt Gemini to summarize ONLY the Tavily results
    prompt = f"""
You are a patient education assistant.

The patient wants to learn about:
{state["user_query"]}

Below are search results retrieved from Tavily.

TAVILY SEARCH RESULTS:
{context}

Instructions:
- Summarize the information using ONLY the Tavily search results provided above.
- Do NOT use outside medical knowledge.
- Do NOT add facts that are not present in the search results.
- Write in simple, patient-friendly language.
- Create exactly 3-4 clear paragraphs.
- Explain the health topic in an educational and easy-to-understand way.
- Include important information supported by the sources.
- Do not diagnose the patient.
- Do not provide personalized medical treatment advice.
- Mention important warning signs or when medical attention may be needed only if they are supported by the Tavily results.

Return ONLY the patient-friendly 3-4 paragraph summary.
"""

    # Generate summary using Gemini
    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt
    )

    # Store Gemini's summary in the state
    state["response"] = response.text

    return state


print("Gemini response node defined successfully.")

Gemini response node defined successfully.


In [9]:
# ============================================================
# Cell 9: Build HealthBot Workflow
# ============================================================

from langgraph.graph import StateGraph, START, END

workflow = StateGraph(HealthBotState)

# Add nodes
workflow.add_node("search", search_medical_information)
workflow.add_node("generate", generate_response)

# Define flow
workflow.add_edge(START, "search")
workflow.add_edge("search", "generate")
workflow.add_edge("generate", END)

# Compile the graph
healthbot_graph = workflow.compile()

print("HealthBot workflow compiled successfully.")

HealthBot workflow compiled successfully.


In [10]:
# ============================================================
# Cell 10: Test Complete HealthBot
# ============================================================

user_query = "What are the common symptoms of dehydration?"

initial_state = {
    "user_query": user_query,
    "context": "",
    "search_results": [],
    "response": ""
}

final_state = healthbot_graph.invoke(initial_state)

print("USER QUERY:")
print(final_state["user_query"])

print("\nHEALTHBOT RESPONSE:")
print(final_state["response"])

USER QUERY:
What are the common symptoms of dehydration?

HEALTHBOT RESPONSE:
Dehydration happens when your body loses more fluids than it takes in, which prevents it from working properly. In adults, common early or mild signs of dehydration often start with feeling very thirsty, sweating less, and noticing dark-colored urine or urinating less often than usual. Other common symptoms include a dry or sticky mouth, dry skin, feeling unusually tired or weak, headaches, a fever, and feeling dizzy or lightheaded, particularly when standing up.

Infants and young children can also become dehydrated, and their symptoms may present somewhat differently. Common signs to look for in young children include a dry mouth and tongue, crying without tears, and being unusually sleepy, drowsy, or irritable. They may also have sunken eyes, a high fever, fewer wet diapers, no wet diapers for three hours or more, or a sunken soft spot (called a fontanelle) on the top of a baby's head.

Dehydration can ran

In [11]:
# ============================================================
# Cell 11: Test HealthBot with Another Query
# ============================================================

user_query = "What are the common symptoms of flu?"

initial_state = {
    "user_query": user_query,
    "context": "",
    "search_results": [],
    "response": ""
}

final_state = healthbot_graph.invoke(initial_state)

print("USER QUERY:")
print(final_state["user_query"])

print("\nHEALTHBOT RESPONSE:")
print(final_state["response"])

USER QUERY:
What are the common symptoms of flu?

HEALTHBOT RESPONSE:
The flu is a contagious respiratory illness caused by the influenza virus. Flu symptoms typically come on suddenly, often within one to three days after you are exposed to the virus. The most common early symptoms include a sudden fever, chills, muscle and body aches, headaches, and feeling very tired or run down. While fever is a primary sign of the flu, it is less common in older adults. 

Along with aches and fever, the flu commonly affects your respiratory system and can sometimes affect your digestive system. Respiratory symptoms can include a sore throat, a runny or blocked nose, sneezing, and a cough that may be dry or chesty. You might also experience a loss of appetite, difficulty sleeping, or tummy issues such as feeling sick, vomiting, diarrhea, and stomach pain. Stomach symptoms like vomiting and diarrhea are more common in children than adults. Children may also become less active or experience ear pain.

In [12]:
# ============================================================
# Cell 12: Test HealthBot with Multiple Queries
# ============================================================

test_queries = [
    "What are the common symptoms of dehydration?",
    "What are the common symptoms of flu?"
]

for query in test_queries:
    print("=" * 70)
    print("USER QUERY:")
    print(query)

    initial_state = {
        "user_query": query,
        "context": "",
        "search_results": [],
        "response": ""
    }

    final_state = healthbot_graph.invoke(initial_state)

    print("\nHEALTHBOT RESPONSE:")
    print(final_state["response"])
    print()

USER QUERY:
What are the common symptoms of dehydration?

HEALTHBOT RESPONSE:
Dehydration occurs when your body loses more fluid than you take in, which prevents it from working properly. In adults, early or mild-to-moderate signs of dehydration often begin with feeling thirsty and having a dry or sticky mouth, dry lips, or a dry tongue. You might also experience a headache, dry skin, or feel unusually tired, weak, dizzy, or lightheaded, especially when standing up. Another common indicator is a change in your urination habits, such as peeing less often, sweating less than usual, or having dark-colored urine.

In infants and young children, symptoms of dehydration can look somewhat different. A dehydrated child may have a dry mouth and tongue, cry without tears, or have a high fever. Caregivers might notice fewer wet diapers—such as no wet diaper for three hours or more—or older children going to the bathroom less frequently. Children may also appear unusually sleepy, drowsy, or irrita

In [13]:
# ============================================================
# Cell 13: Extend HealthBot State for Quiz and Session Control
# ============================================================

class HealthBotState(TypedDict, total=False):
    user_query: str
    context: str
    search_results: List[Dict[str, Any]]
    response: str

    # Quiz fields
    quiz: str
    user_answers: str
    score: int
    feedback: str

    # Session control
    continue_session: bool


print("HealthBot state extended for quiz and session control successfully.")

HealthBot state extended for quiz and session control successfully.


In [14]:
# ============================================================
# Cell 14: Quiz Generation Node
# ============================================================

def generate_quiz(state: HealthBotState) -> HealthBotState:

    summary = state["response"]

    prompt = f"""
You are a medical education assistant.

Based ONLY on the SUMMARY provided below, create ONE
multiple-choice comprehension question.

Rules:
- Create exactly ONE question.
- Provide exactly 4 options: A, B, C, D.
- Only ONE option should be correct.
- The question must be answerable using ONLY the summary.
- Do not use outside medical knowledge.
- Do not provide the correct answer.
- Keep the question simple and educational.

SUMMARY:
{summary}

Return ONLY the question and four options.
"""

    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt
    )

    state["quiz"] = response.text

    return state


print("Quiz generation node defined successfully.")

Quiz generation node defined successfully.


In [15]:
# ============================================================
# Cell 15: Add Quiz Node to LangGraph
# ============================================================

workflow.add_node("quiz", generate_quiz)

# Current flow:
# START → search → generate → quiz → END

workflow.add_edge("generate", "quiz")
workflow.add_edge("quiz", END)

# Recompile the graph
healthbot_graph = workflow.compile()

print("Quiz node added and workflow recompiled successfully.")

Adding a node to a graph that has already been compiled. This will not be reflected in the compiled graph.
Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.
Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.


Quiz node added and workflow recompiled successfully.


In [16]:
# ============================================================
# Cell 16: Collect User's Quiz Answer
# ============================================================

def collect_answer(state: HealthBotState) -> HealthBotState:
    print("\n" + "=" * 60)
    print("QUIZ QUESTION:")
    print(state["quiz"])
    print("=" * 60)

    user_answer = input("Your answer: ")

    state["user_answers"] = user_answer

    return state


print("Quiz answer collection node defined successfully.")

Quiz answer collection node defined successfully.


In [17]:
# ============================================================
# Cell 17: Grade User's Quiz Answer
# ============================================================

def grade_answer(state: HealthBotState) -> HealthBotState:

    summary = state["response"]
    quiz = state["quiz"]
    user_answer = state["user_answers"]

    prompt = f"""
You are grading a health education comprehension quiz.

IMPORTANT RULES:

- Use ONLY the medical information provided in the SUMMARY below.
- Do NOT use outside medical knowledge.
- Do NOT introduce facts that are not present in the summary.
- Grade the user's answer based only on whether it matches the information
  contained in the summary.
- Give a letter grade: A, B, C, D, or F.
- Explain clearly why the grade was given.
- Your explanation MUST refer to specific information from the summary.
- The citation must quote or closely reference the exact relevant information
  from the SUMMARY.
- Keep the explanation patient-friendly.

SUMMARY:

{summary}

QUIZ QUESTION:

{quiz}

USER'S ANSWER:

{user_answer}

Return the result EXACTLY in this format:

GRADE: <A/B/C/D/F>

FEEDBACK:
<Explain why the answer received this grade using only information from the summary.>

CITATION FROM SUMMARY:
<Quote or closely reference the specific sentence or information from the summary that supports the grade.>
"""

    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt
    )

    feedback = response.text

    state["feedback"] = feedback

    return state


print("Quiz grading node defined successfully.")

Quiz grading node defined successfully.


In [18]:
# ============================================================
# Cell 18: Add Answer Collection and Grading Nodes
# ============================================================

workflow.add_node("collect_answer", collect_answer)
workflow.add_node("grade", grade_answer)

print("Answer collection and grading nodes added successfully.")

Adding a node to a graph that has already been compiled. This will not be reflected in the compiled graph.
Adding a node to a graph that has already been compiled. This will not be reflected in the compiled graph.


Answer collection and grading nodes added successfully.


In [19]:
# ============================================================
# Cell 19: Build the Complete HealthBot Workflow
# ============================================================

from langgraph.graph import StateGraph, START, END


# ------------------------------------------------------------
# Patient interaction nodes
# ------------------------------------------------------------

def ask_topic(state: HealthBotState) -> HealthBotState:

    print("\n" + "=" * 70)
    print("WELCOME TO HEALTHBOT")
    print("Learn about a health topic and test your understanding.")
    print("=" * 70)

    topic = input(
        "What health topic or medical condition would you like to learn about? "
    ).strip()

    # --------------------------------------------------------
    # Reset previous topic information
    # This is required when starting a new topic.
    # --------------------------------------------------------

    state["user_query"] = topic
    state["context"] = ""
    state["search_results"] = []
    state["response"] = ""
    state["quiz"] = ""
    state["user_answers"] = ""
    state["feedback"] = ""
    state["continue_session"] = False

    return state


def present_summary(state: HealthBotState) -> HealthBotState:

    print("\n" + "=" * 70)
    print("HEALTH INFORMATION")
    print("=" * 70)

    print(state["response"])

    print("=" * 70)

    input(
        "Press Enter when you have finished reading the information... "
    )

    return state


def ready_for_quiz(state: HealthBotState) -> HealthBotState:

    print("\n" + "=" * 70)
    print("COMPREHENSION CHECK")
    print("=" * 70)

    input(
        "Press Enter when you are ready to take the quiz... "
    )

    return state


def present_feedback(state: HealthBotState) -> HealthBotState:

    print("\n" + "=" * 70)
    print("QUIZ RESULT")
    print("=" * 70)

    print(state["feedback"])

    print("=" * 70)

    return state


def ask_continue(state: HealthBotState) -> HealthBotState:

    choice = input(
        "\nWould you like to learn about another health topic? "
        "(yes/no): "
    ).strip().lower()

    if choice in ["yes", "y"]:
        state["continue_session"] = True

        print("\nStarting a new HealthBot topic...")

    else:
        state["continue_session"] = False

        print("\nEnding HealthBot session...")

    return state


# ------------------------------------------------------------
# Create a fresh workflow
# ------------------------------------------------------------

workflow = StateGraph(HealthBotState)


# ------------------------------------------------------------
# Add all nodes
# ------------------------------------------------------------

workflow.add_node("ask_topic", ask_topic)

workflow.add_node(
    "search",
    search_medical_information
)

workflow.add_node(
    "generate",
    generate_response
)

workflow.add_node(
    "present_summary",
    present_summary
)

workflow.add_node(
    "ready_for_quiz",
    ready_for_quiz
)

workflow.add_node(
    "quiz",
    generate_quiz
)

workflow.add_node(
    "collect_answer",
    collect_answer
)

workflow.add_node(
    "grade",
    grade_answer
)

workflow.add_node(
    "present_feedback",
    present_feedback
)

workflow.add_node(
    "ask_continue",
    ask_continue
)


# ------------------------------------------------------------
# Define the complete flow
# ------------------------------------------------------------

workflow.add_edge(
    START,
    "ask_topic"
)

workflow.add_edge(
    "ask_topic",
    "search"
)

workflow.add_edge(
    "search",
    "generate"
)

workflow.add_edge(
    "generate",
    "present_summary"
)

workflow.add_edge(
    "present_summary",
    "ready_for_quiz"
)

workflow.add_edge(
    "ready_for_quiz",
    "quiz"
)

workflow.add_edge(
    "quiz",
    "collect_answer"
)

workflow.add_edge(
    "collect_answer",
    "grade"
)

workflow.add_edge(
    "grade",
    "present_feedback"
)

workflow.add_edge(
    "present_feedback",
    "ask_continue"
)


# ------------------------------------------------------------
# Conditional routing after quiz result
# ------------------------------------------------------------

def route_after_continue(state: HealthBotState):

    if state.get("continue_session", False):

        return "ask_topic"

    return END


workflow.add_conditional_edges(
    "ask_continue",
    route_after_continue,
    {
        "ask_topic": "ask_topic",
        END: END
    }
)


# ------------------------------------------------------------
# Compile the complete graph
# ------------------------------------------------------------

healthbot_graph = workflow.compile()

print(
    "Complete HealthBot workflow compiled successfully."
)

Complete HealthBot workflow compiled successfully.


In [20]:
# ============================================================
# Cell 20: Run HealthBot
# ============================================================

initial_state = {
    "user_query": "",
    "context": "",
    "search_results": [],
    "response": "",
    "quiz": "",
    "user_answers": "",
    "feedback": "",
    "continue_session": False
}

result = healthbot_graph.invoke(initial_state)

print("\n" + "=" * 70)
print("HEALTHBOT SESSION COMPLETED")
print("=" * 70)


WELCOME TO HEALTHBOT
Learn about a health topic and test your understanding.

HEALTH INFORMATION
Sexual expression and activity, including both solo and joint masturbation, are important parts of overall sexual well-being. Discussing sexual health in the context of aging, disease, or medical interventions can help support a person's or couple's quality of life. Maintaining sexual expression can positively impact your well-being, self-image, and emotional intimacy.

However, some individuals may experience painful intercourse, also known as dyspareunia. Pain during sex can affect people of any age, though it is more common in women past menopause due to decreased estrogen levels, which can reduce vaginal lubrication. Painful sex can also be linked to emotional factors, lack of foreplay, or certain medications, including antidepressants, high blood pressure medicines, sedatives, antihistamines, and some birth control pills. 

Symptoms of painful intercourse may include sharp pain during